In [2]:
import pandas as pd

In [4]:
import os
os.getcwd()

'/Users/sharvarigangamwar/Downloads/Finance Project'

In [5]:
os.listdir()

['.ipynb_checkpoints', 'budget_tracker.ipynb', 'sample_transactions (1).csv']

In [9]:
df = pd.read_csv("sample_transactions (1).csv")

In [10]:
os.rename("sample_transactions (1).csv", "sample_transactions.csv")

In [11]:
df.shape

(350, 3)

In [12]:
df.head(10)

,Date,Description,Amount
0,01/01/2025,Rent Payment - Sunrise Apartments,-1450.00
1,01/01/2025,Employer Payroll Direct Dep,2650.00
2,01/01/2025,Local Pizza Co,-25.13
3,01/03/2025,Geico Insurance,-135.25
4,01/04/2025,Planet Fitness,-24.99
5,01/05/2025,Netflix.com,-15.49
6,01/05/2025,Airbnb,-257.17
7,01/05/2025,Ticketmaster,-81.98
8,01/06/2025,Best Buy,-292.19
9,01/06/2025,Trader Joe's,-69.22


In [13]:
df.columns.tolist()

['Date', 'Description', 'Amount']

In [14]:
df.dtypes

Date            object
Description     object
Amount         float64
dtype: object

In [15]:
df.isnull().sum()

Date           0
Description    0
Amount         0
dtype: int64

In [16]:
df["Date"].head()
df["Date"].min(), df["Date"].max()

('01/01/2025', '06/29/2025')

In [17]:
df["Amount"].describe()

count     350.000000
mean        5.486714
std       557.918340
min     -1450.000000
25%       -90.900000
50%       -45.250000
75%       -20.827500
max      2650.000000
Name: Amount, dtype: float64

In [18]:
df["Description"].nunique()
df["Description"].unique()[:20]

array(['Rent Payment - Sunrise Apartments', 'Employer Payroll Direct Dep',
       'Local Pizza Co', 'Geico Insurance', 'Planet Fitness',
       'Netflix.com', 'Airbnb', 'Ticketmaster', 'Best Buy',
       "Trader Joe's", 'The Cheesecake Factory', 'Spotify USA', 'H&M',
       'REI', 'Comcast Xfinity', 'Chase Credit Card Payment',
       'CVS Pharmacy', 'Lyft', 'PG&E Electric', 'PetSmart'], dtype=object)

In [19]:
df["Date"] = pd.to_datetime(df["Date"])
df["Date"].head()
df.dtypes

Date           datetime64[ns]
Description            object
Amount                float64
dtype: object

In [20]:
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")

In [21]:
df["Amount"] = df["Amount"].astype(float)

In [22]:
df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

In [23]:
df["Month"] = df["Date"].dt.to_period("M").astype(str)
df["Month"].head()

0    2025-01
1    2025-01
2    2025-01
3    2025-01
4    2025-01
Name: Month, dtype: object

In [24]:
df["Type"] = df["Amount"].apply(lambda x: "Income" if x > 0 else "Expense")
df["Type"].value_counts()

Type
Expense    333
Income      17
Name: count, dtype: int64

In [25]:
df.dtypes
df.head()

,Date,Description,Amount,Month,Type
0,2025-01-01,Rent Payment - Sunrise Apartments,-1450.00,2025-01,Expense
1,2025-01-01,Employer Payroll Direct Dep,2650.00,2025-01,Income
2,2025-01-01,Local Pizza Co,-25.13,2025-01,Expense
3,2025-01-03,Geico Insurance,-135.25,2025-01,Expense
4,2025-01-04,Planet Fitness,-24.99,2025-01,Expense


In [26]:
CATEGORY_RULES = {
    "Housing": ["rent", "apartments", "mortgage"],
    "Utilities": ["comcast", "xfinity", "pg&e", "electric", "verizon"],
    "Subscriptions": ["netflix", "spotify", "amazon prime", "nytimes"],
    "Insurance": ["geico", "insurance"],
    "Debt Payments": ["credit card payment", "student loan"],
    "Groceries": ["trader joe", "safeway", "whole foods"],
    "Dining Out": ["chipotle", "starbucks", "sushi", "doordash", "pizza", "cheesecake factory"],
    "Transportation": ["shell", "chevron", "uber", "lyft", "transit"],
    "Shopping": ["target", "amazon.com", "best buy", "h&m", "rei"],
    "Health & Fitness": ["cvs", "walgreens", "kaiser", "planet fitness"],
    "Entertainment": ["amc", "steam", "ticketmaster", "bar & grill"],
    "Travel": ["southwest", "airlines", "airbnb", "marriott"],
    "Pets": ["petsmart"],
    "Cash & Misc": ["atm withdrawal", "venmo"],
    "Income": ["payroll", "direct dep", "freelance"],
}

In [28]:
def categorize(description):
    desc = description.lower()
    for category, keywords in CATEGORY_RULES.items():
        if any(kw in desc for kw in keywords):
            return category
    return "Uncategorized"
    

In [29]:
df["Category"] = df["Description"].apply(categorize)
df["Category"].value_counts()

Category
Shopping            50
Dining Out          45
Health & Fitness    35
Transportation      32
Travel              27
Subscriptions       24
Groceries           22
Entertainment       21
Cash & Misc         21
Utilities           18
Income              17
Debt Payments       13
Pets                13
Housing              6
Insurance            6
Name: count, dtype: int64

In [30]:
df[df["Category"] == "Uncategorized"]["Description"].unique()

array([], dtype=object)

In [31]:
exp = df[df["Amount"] < 0].copy()
exp["AbsAmount"] = exp["Amount"].abs()

In [32]:
grouped = exp.groupby("Description").agg(
    occurrences=("Amount", "count"),
    avg_amount=("AbsAmount", "mean"),
    months_seen=("Month", "nunique"),
).reset_index()

grouped.head()

,Description,occurrences,avg_amount,months_seen
0,AMC Theatres,3,19.726667,3
1,ATM Withdrawal,12,62.175833,6
2,Airbnb,8,210.853750,5
3,Amazon Prime,6,14.990000,6
4,Amazon.com,18,82.932222,6


In [33]:
recurring = grouped[grouped["months_seen"] >= 3].copy()
recurring = recurring.sort_values("months_seen", ascending=False)
recurring

,Description,occurrences,avg_amount,months_seen
21,Netflix.com,6,15.490000,6
33,Student Loan Servicer,6,220.000000,6
20,NYTimes Subscription,6,17.000000,6
24,Planet Fitness,6,24.990000,6
16,Local Bar & Grill,8,42.557500,6
26,Rent Payment - Sunrise Apartments,6,1450.000000,6
30,Spotify USA,6,11.990000,6
13,Geico Insurance,6,136.650000,6
31,Starbucks,8,7.623750,6
22,PG&E Electric,6,97.450000,6


In [34]:
recurring["est_annual_cost"] = recurring["avg_amount"] * 12
recurring = recurring.sort_values("est_annual_cost", ascending=False)
recurring[["Description", "occurrences", "avg_amount", "est_annual_cost"]]

,Description,occurrences,avg_amount,est_annual_cost
26,Rent Payment - Sunrise Apartments,6,1450.000000,17400.000000
8,Chase Credit Card Payment,7,371.337143,4456.045714
29,Southwest Airlines,8,281.996250,3383.955000
19,Marriott Hotels,11,222.250909,2667.010909
33,Student Loan Servicer,6,220.000000,2640.000000
2,Airbnb,8,210.853750,2530.245000
6,Best Buy,8,157.615000,1891.380000
13,Geico Insurance,6,136.650000,1639.800000
25,REI,7,111.141429,1333.697143
22,PG&E Electric,6,97.450000,1169.400000


In [35]:
recurring_std = exp.groupby("Description")["AbsAmount"].std().reset_index()
recurring_std.columns = ["Description", "amount_std"]
recurring = recurring.merge(recurring_std, on="Description")
true_subscriptions = recurring[recurring["amount_std"] < 5]  # amount barely varies
true_subscriptions

,Description,occurrences,avg_amount,months_seen,est_annual_cost,amount_std
0,Rent Payment - Sunrise Apartments,6,1450.000000,6,17400.000000,0.000000
4,Student Loan Servicer,6,220.000000,6,2640.000000,0.000000
7,Geico Insurance,6,136.650000,6,1639.800000,1.335665
10,Comcast Xfinity,6,90.013333,6,1080.160000,2.326617
13,Verizon Wireless,6,81.525000,6,978.300000,1.922943
31,Local Pizza Co,8,25.431250,5,305.175000,4.598930
32,Planet Fitness,6,24.990000,6,299.880000,0.000000
34,AMC Theatres,3,19.726667,3,236.720000,1.850468
35,Lyft,9,17.125556,4,205.506667,4.788265
36,NYTimes Subscription,6,17.000000,6,204.000000,0.000000


In [36]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

In [37]:
wb = Workbook()
tx = wb.active
tx.title = "Transactions"

headers = ["Date", "Description", "Amount", "Category", "Type", "Month"]
tx.append(headers)

for _, r in df.iterrows():
    tx.append([
        r["Date"].strftime("%Y-%m-%d"),
        r["Description"],
        r["Amount"],
        r["Category"],
        r["Type"],
        r["Month"],
    ])

In [38]:
HEADER_FILL = PatternFill("solid", start_color="1F4E78")
HEADER_FONT = Font(bold=True, color="FFFFFF")

for c in range(1, len(headers) + 1):
    cell = tx.cell(row=1, column=c)
    cell.font = HEADER_FONT
    cell.fill = HEADER_FILL
    cell.alignment = Alignment(horizontal="center")

In [39]:
n_rows = len(df)
for row in range(2, n_rows + 2):
    tx.cell(row=row, column=3).number_format = '$#,##0.00;($#,##0.00)'

In [40]:
tx.freeze_panes = "A2"
tx.auto_filter.ref = f"A1:F{n_rows + 1}"

widths = [12, 34, 12, 18, 10, 10]
for i, w in enumerate(widths, start=1):
    tx.column_dimensions[get_column_letter(i)].width = w

In [41]:
wb.save("Budget_Dashboard.xlsx")

In [42]:
months = sorted(df["Month"].unique())
expense_categories = sorted(df.loc[df["Category"] != "Income", "Category"].unique())

months, expense_categories

(['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06'],
 ['Cash & Misc',
  'Debt Payments',
  'Dining Out',
  'Entertainment',
  'Groceries',
  'Health & Fitness',
  'Housing',
  'Insurance',
  'Pets',
  'Shopping',
  'Subscriptions',
  'Transportation',
  'Travel',
  'Utilities'])

In [43]:
ms = wb.create_sheet("Monthly Summary")

header_row = 3
ms.cell(row=header_row, column=1, value="Category").font = HEADER_FONT
ms.cell(row=header_row, column=1).fill = HEADER_FILL

for j, m in enumerate(months, start=2):
    c = ms.cell(row=header_row, column=j, value=m)
    c.font = HEADER_FONT
    c.fill = HEADER_FILL

total_col = len(months) + 2
c = ms.cell(row=header_row, column=total_col, value="Total")
c.font = HEADER_FONT
c.fill = HEADER_FILL

In [44]:
LAST = n_rows + 1  # from Step 6, n_rows = len(df)

for i, cat in enumerate(expense_categories, start=1):
    r = header_row + i
    ms.cell(row=r, column=1, value=cat)
    for j, m in enumerate(months, start=2):
        col_letter = get_column_letter(j)
        formula = (
            f'=-SUMIFS(Transactions!$C$2:$C${LAST},'
            f'Transactions!$D$2:$D${LAST},$A{r},'
            f'Transactions!$F$2:$F${LAST},{col_letter}${header_row})'
        )
        cell = ms.cell(row=r, column=j, value=formula)
        cell.number_format = '$#,##0'

In [46]:
total_letter_start = get_column_letter(2)
total_letter_end = get_column_letter(len(months) + 1)

for i, cat in enumerate(expense_categories, start=1):
    r = header_row + i
    cell = ms.cell(row=r, column=total_col, value=f"=SUM({total_letter_start}{r}:{total_letter_end}{r})")
    cell.number_format = '$#,##0'

In [47]:
wb.save("Budget_Dashboard.xlsx")

In [48]:
total_row = header_row + len(expense_categories) + 1
ms.cell(row=total_row, column=1, value="Total Spending").font = BOLD if 'BOLD' in dir() else Font(bold=True)

for j in range(2, total_col + 1):
    col_letter = get_column_letter(j)
    first_data_row = header_row + 1
    last_data_row = header_row + len(expense_categories)
    cell = ms.cell(row=total_row, column=j, value=f"=SUM({col_letter}{first_data_row}:{col_letter}{last_data_row})")
    cell.number_format = '$#,##0'
    cell.font = Font(bold=True)

In [49]:
iv_row = total_row + 3
ms.cell(row=iv_row, column=1, value="Income vs. Expenses vs. Savings").font = Font(bold=True, size=12)

hdr2 = iv_row + 1
ms.cell(row=hdr2, column=1, value="Metric").font = HEADER_FONT
ms.cell(row=hdr2, column=1).fill = HEADER_FILL

for j, m in enumerate(months, start=2):
    c = ms.cell(row=hdr2, column=j, value=m)
    c.font = HEADER_FONT
    c.fill = HEADER_FILL

In [50]:
income_row = hdr2 + 1
expense_row = hdr2 + 2
net_row = hdr2 + 3
rate_row = hdr2 + 4

ms.cell(row=income_row, column=1, value="Income")
ms.cell(row=expense_row, column=1, value="Expenses")
ms.cell(row=net_row, column=1, value="Net Savings").font = Font(bold=True)
ms.cell(row=rate_row, column=1, value="Savings Rate").font = Font(bold=True)

for j, m in enumerate(months, start=2):
    col_letter = get_column_letter(j)

    # Income: pull straight from Transactions where Type = "Income"
    inc_formula = (
        f'=SUMIFS(Transactions!$C$2:$C${LAST},'
        f'Transactions!$E$2:$E${LAST},"Income",'
        f'Transactions!$F$2:$F${LAST},{col_letter}${header_row})'
    )
    ms.cell(row=income_row, column=j, value=inc_formula).number_format = '$#,##0'

    # Expenses: just reference the Total Spending row we already built
    ms.cell(row=expense_row, column=j, value=f"={col_letter}{total_row}").number_format = '$#,##0'

    # Net savings: income minus expenses
    net_cell = ms.cell(row=net_row, column=j, value=f"={col_letter}{income_row}-{col_letter}{expense_row}")
    net_cell.number_format = '$#,##0'
    net_cell.font = Font(bold=True)

    # Savings rate: net / income, as a percent
    rate_cell = ms.cell(row=rate_row, column=j,
                         value=f"=IF({col_letter}{income_row}=0,0,{col_letter}{net_row}/{col_letter}{income_row})")
    rate_cell.number_format = '0.0%'
    rate_cell.font = Font(bold=True)

In [51]:
wb.save("Budget_Dashboard.xlsx")

In [52]:
rs = wb.create_sheet("Recurring Costs")
rs["A1"] = "Detected Recurring Charges (3+ months)"
rs["A1"].font = Font(bold=True, size=14)

rheader = ["Description", "Occurrences", "Avg Amount", "Est. Annual Cost"]
for j, h in enumerate(rheader, start=1):
    c = rs.cell(row=3, column=j, value=h)
    c.font = HEADER_FONT
    c.fill = HEADER_FILL

for i, row in recurring.reset_index(drop=True).iterrows():
    r = 4 + i
    rs.cell(row=r, column=1, value=row["Description"])
    rs.cell(row=r, column=2, value=int(row["occurrences"]))
    rs.cell(row=r, column=3, value=round(float(row["avg_amount"]), 2)).number_format = '$#,##0.00'
    rs.cell(row=r, column=4, value=round(float(row["est_annual_cost"]), 2)).number_format = '$#,##0'

rs.column_dimensions["A"].width = 34

In [53]:
from openpyxl.chart import PieChart, BarChart, LineChart, Reference

dash = wb.create_sheet("Dashboard", 0)  # 0 = insert as first tab
dash["A1"] = "Personal Budget Dashboard"
dash["A1"].font = Font(bold=True, size=18)

In [54]:
pie = PieChart()
pie.title = "Total Spending by Category"

cats_ref = Reference(ms, min_col=1, min_row=header_row + 1, max_row=header_row + len(expense_categories))
data_ref = Reference(ms, min_col=total_col, min_row=header_row, max_row=header_row + len(expense_categories))

pie.add_data(data_ref, titles_from_data=True)
pie.set_categories(cats_ref)
pie.height = 9
pie.width = 15
dash.add_chart(pie, "A4")

In [55]:
bar = BarChart()
bar.type = "col"
bar.title = "Income vs. Expenses by Month"
bar.y_axis.title = "USD"

data_ref2 = Reference(ms, min_col=1, max_col=len(months) + 1, min_row=income_row, max_row=expense_row)
cats_ref2 = Reference(ms, min_col=2, max_col=len(months) + 1, min_row=hdr2)

bar.add_data(data_ref2, titles_from_data=True, from_rows=True)
bar.set_categories(cats_ref2)
bar.height = 9
bar.width = 15
dash.add_chart(bar, "J4")

In [56]:
line = LineChart()
line.title = "Net Savings Trend"
line.y_axis.title = "USD"

data_ref3 = Reference(ms, min_col=1, max_col=len(months) + 1, min_row=net_row)
cats_ref3 = Reference(ms, min_col=2, max_col=len(months) + 1, min_row=hdr2)

line.add_data(data_ref3, titles_from_data=True, from_rows=True)
line.set_categories(cats_ref3)
line.height = 9
line.width = 15
dash.add_chart(line, "A22")

In [57]:
wb.save("Budget_Dashboard.xlsx")

In [59]:
from categorize import load_and_process, detect_recurring
test_df = load_and_process("sample_transactions.csv")
test_df.head()

,Date,Description,Amount,Category,Type,Month
0,2025-01-01,Rent Payment - Sunrise Apartments,-1450.00,Housing,Expense,2025-01
1,2025-01-01,Employer Payroll Direct Dep,2650.00,Income,Income,2025-01
2,2025-01-01,Local Pizza Co,-25.13,Dining Out,Expense,2025-01
3,2025-01-03,Geico Insurance,-135.25,Insurance,Expense,2025-01
4,2025-01-04,Planet Fitness,-24.99,Health & Fitness,Expense,2025-01


In [60]:
pip install streamlit plotly

Note: you may need to restart the kernel to use updated packages.


In [1]:
streamlit run app.py

SyntaxError: invalid syntax (3737097518.py, line 1)